Smart Park Seattle — MVSTGCN End-to-End Pipeline

In [1]:
from dataclasses import dataclass
from datetime import datetime, timezone
import json
import boto3
import pandas as pd

s3 = boto3.client("s3")


@dataclass
class ServingBuildConfig:
    bucket: str = "smart-park-seattle"

    input_prefix_y15: str = "parking_v2/preds/final/year=2023/target=y_15/"
    input_prefix_y30: str = "parking_v2/preds/final/year=2023/target=y_30/"

    output_prefix_y15: str = "parking_v2/serving/current/target=y_15/"
    output_prefix_y30: str = "parking_v2/serving/current/target=y_30/"

    nodes_key: str = "parking_v2/graphs/year=2023/nodes.csv"

    timezone: str = "America/Los_Angeles"

    col_time: str = "ts15_utc"
    col_node_id: str = "sourceelementkey"
    col_p15: str = "p15"
    col_p30: str = "p30"

    node_id_col: str = "sourceelementkey"
    node_name_col: str | None = "blockfacename"
    node_address_col: str | None = None
    node_lat_col: str = "lat"
    node_lng_col: str = "lon"
    node_area_col: str | None = "area"
    node_rate_col: str | None = "rate_mean"
    node_total_spots_col: str | None = "space_count_max"
    node_risk_col: str | None = None

In [2]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def list_s3_keys(bucket: str, prefix: str, suffix: str | None = None) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if suffix is None or key.endswith(suffix):
                keys.append(key)

    return sorted(keys)


def read_s3_csv_gz(bucket: str, key: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj["Body"], compression="gzip")


def read_s3_csv(bucket: str, key: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj["Body"])


def write_json_to_s3(bucket: str, key: str, obj: dict):
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=json.dumps(obj, ensure_ascii=False).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"wrote s3://{bucket}/{key}")


def read_json_from_s3(bucket: str, key: str) -> dict:
    obj = s3.get_object(Bucket=bucket, Key=key)
    return json.loads(obj["Body"].read().decode("utf-8"))


def s3_object_exists(bucket: str, key: str) -> bool:
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except Exception:
        return False


def num_or_none(val):
    if pd.isna(val):
        return None
    return float(val)


def int_or_none(val):
    if pd.isna(val):
        return None
    return int(val)


def safe_get(row: pd.Series, col: str | None, default=None):
    if not col or col not in row:
        return default
    val = row[col]
    if pd.isna(val):
        return default
    return val

In [3]:
def load_nodes_metadata(cfg: ServingBuildConfig) -> pd.DataFrame:
    nodes_df = read_s3_csv(cfg.bucket, cfg.nodes_key).copy()

    required = [cfg.node_id_col, cfg.node_lat_col, cfg.node_lng_col]
    missing = [c for c in required if c not in nodes_df.columns]
    if missing:
        raise ValueError(f"nodes.csv missing required columns: {missing}")

    keep_cols = [cfg.node_id_col, cfg.node_lat_col, cfg.node_lng_col]

    optional_cols = [
        cfg.node_name_col,
        cfg.node_address_col,
        cfg.node_area_col,
        cfg.node_rate_col,
        cfg.node_total_spots_col,
        cfg.node_risk_col,
    ]
    for c in optional_cols:
        if c and c in nodes_df.columns and c not in keep_cols:
            keep_cols.append(c)

    meta = nodes_df[keep_cols].copy()
    meta[cfg.node_id_col] = meta[cfg.node_id_col].astype(str)

    return meta


def row_to_item(row: pd.Series, cfg: ServingBuildConfig, target: str) -> dict:
    node_id = str(row[cfg.col_node_id])

    p15 = None
    p30 = None

    if target == "y_15" and cfg.col_p15 in row:
        p15 = num_or_none(row[cfg.col_p15])
    if target == "y_30" and cfg.col_p30 in row:
        p30 = num_or_none(row[cfg.col_p30])

    # fallback so UI can still read both fields
    if p15 is None and p30 is not None:
        p15 = p30
    if p30 is None and p15 is not None:
        p30 = p15

    return {
        "id": node_id,
        "name": str(safe_get(row, cfg.node_name_col, node_id)),
        "address": "",
        "lat": float(row[cfg.node_lat_col]),
        "lng": float(row[cfg.node_lng_col]),
        "ratePerHour": num_or_none(row[cfg.node_rate_col]) if cfg.node_rate_col and cfg.node_rate_col in row else None,
        "p15": p15,
        "p30": p30,
        "riskScore": num_or_none(row[cfg.node_risk_col]) if cfg.node_risk_col and cfg.node_risk_col in row else None,
        "totalSpots": int_or_none(row[cfg.node_total_spots_col]) if cfg.node_total_spots_col and cfg.node_total_spots_col in row else None,
        "area": safe_get(row, cfg.node_area_col, None),
    }

In [4]:
def build_date_payload(
    day_df: pd.DataFrame,
    date: str,
    cfg: ServingBuildConfig,
    target: str,
    generated_at: str,
) -> dict:
    slot_times = sorted(day_df["slot_time"].unique().tolist())

    slots = []
    for slot_time in slot_times:
        slot_df = day_df[day_df["slot_time"] == slot_time]
        items = [row_to_item(row, cfg, target) for _, row in slot_df.iterrows()]
        slots.append({
            "time": slot_time,
            "items": items,
        })

    return {
        "date": date,
        "timezone": cfg.timezone,
        "generated_at": generated_at,
        "slots": slots,
    }

In [5]:
def build_serving_files_for_target_lowmem(
    cfg: ServingBuildConfig,
    input_prefix: str,
    output_prefix: str,
    target: str,   # "y_15" or "y_30"
    limit_files: int | None = None,
):
    keys = list_s3_keys(cfg.bucket, input_prefix, suffix="pred.csv.gz")
    if not keys:
        raise RuntimeError(f"No input files found under s3://{cfg.bucket}/{input_prefix}")

    if limit_files is not None:
        keys = keys[:limit_files]
        print(f"TEST MODE - only processing first {len(keys)} files")

    print(f"Found {len(keys)} prediction files for {target}")

    meta_df = load_nodes_metadata(cfg)
    generated_at = utc_now_iso()
    available_dates = set()

    for i, key in enumerate(keys, start=1):
        print(f"[{i}/{len(keys)}] reading {key}")

        df = read_s3_csv_gz(cfg.bucket, key)
        if df.empty:
            continue

        required = [cfg.col_time, cfg.col_node_id]
        if target == "y_15":
            required.append(cfg.col_p15)
        else:
            required.append(cfg.col_p30)

        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"{key} missing required columns: {missing}")

        df[cfg.col_node_id] = df[cfg.col_node_id].astype(str)

        df = df.merge(
            meta_df,
            left_on=cfg.col_node_id,
            right_on=cfg.node_id_col,
            how="left"
        )

        df[cfg.col_time] = pd.to_datetime(df[cfg.col_time], utc=True, errors="coerce")
        df = df[df[cfg.col_time].notna()].copy()
        df = df[df[cfg.node_lat_col].notna() & df[cfg.node_lng_col].notna()].copy()

        if df.empty:
            continue

        ts_local = df[cfg.col_time].dt.tz_convert(cfg.timezone)
        df["seattle_date"] = ts_local.dt.strftime("%Y-%m-%d")
        df["slot_time"] = ts_local.dt.strftime("%Y-%m-%dT%H:%M:%S%z")
        df["slot_time"] = df["slot_time"].str.replace(r"([+-]\d{2})(\d{2})$", r"\1:\2", regex=True)

        for date, day_df in df.groupby("seattle_date"):
            payload = build_date_payload(day_df, date, cfg, target, generated_at)

            out_key = f"{output_prefix}by-date/{date}.json"
            write_json_to_s3(cfg.bucket, out_key, payload)
            available_dates.add(date)

        del df

    available_dates = sorted(available_dates)
    if not available_dates:
        raise RuntimeError("No date payloads were generated")

    default_date = available_dates[0]

    manifest = {
        "default_date": default_date,
        "available_dates": available_dates,
        "generated_at": generated_at,
        "version": "v1",
    }
    write_json_to_s3(cfg.bucket, f"{output_prefix}manifest.json", manifest)

    latest_key = f"{output_prefix}by-date/{default_date}.json"
    latest_payload = read_json_from_s3(cfg.bucket, latest_key)
    write_json_to_s3(cfg.bucket, f"{output_prefix}latest.json", latest_payload)

    print(f"Finished building serving files for {target}")

In [6]:
cfg = ServingBuildConfig(
    bucket="smart-park-seattle",

    input_prefix_y15="parking_v2/preds/final/year=2023/target=y_15/",
    input_prefix_y30="parking_v2/preds/final/year=2023/target=y_30/",

    output_prefix_y15="parking_v2/serving/current/target=y_15/",
    output_prefix_y30="parking_v2/serving/current/target=y_30/",

    nodes_key="parking_v2/graphs/year=2023/nodes.csv",
    timezone="America/Los_Angeles",

    col_time="ts15_utc",
    col_node_id="sourceelementkey",
    col_p15="p15",
    col_p30="p30",

    node_id_col="sourceelementkey",
    node_name_col="blockfacename",
    node_address_col=None,
    node_lat_col="lat",
    node_lng_col="lon",
    node_area_col="area",
    node_rate_col="rate_mean",
    node_total_spots_col="space_count_max",
    node_risk_col=None,
)

In [7]:
build_serving_files_for_target_lowmem(
    cfg,
    cfg.input_prefix_y15,
    cfg.output_prefix_y15,
    "y_15",
    limit_files=2,
)

TEST MODE - only processing first 2 files
Found 2 prediction files for y_15
[1/2] reading parking_v2/preds/final/year=2023/target=y_15/week=2023-01-02/pred.csv.gz
wrote s3://smart-park-seattle/parking_v2/serving/current/target=y_15/by-date/2023-01-02.json
wrote s3://smart-park-seattle/parking_v2/serving/current/target=y_15/by-date/2023-01-03.json
wrote s3://smart-park-seattle/parking_v2/serving/current/target=y_15/by-date/2023-01-04.json
wrote s3://smart-park-seattle/parking_v2/serving/current/target=y_15/by-date/2023-01-05.json
wrote s3://smart-park-seattle/parking_v2/serving/current/target=y_15/by-date/2023-01-06.json
wrote s3://smart-park-seattle/parking_v2/serving/current/target=y_15/by-date/2023-01-07.json
wrote s3://smart-park-seattle/parking_v2/serving/current/target=y_15/by-date/2023-01-08.json
[2/2] reading parking_v2/preds/final/year=2023/target=y_15/week=2023-01-09/pred.csv.gz
wrote s3://smart-park-seattle/parking_v2/serving/current/target=y_15/by-date/2023-01-09.json
wrote 

In [8]:
out_keys = list_s3_keys(
    "smart-park-seattle",
    "parking_v2/serving/current/target=y_15/"
)

print(f"Found {len(out_keys)} output keys")
for k in out_keys[:50]:
    print(k)

Found 352 output keys
parking_v2/serving/current/target=y_15/by-date/2023-01-02.json
parking_v2/serving/current/target=y_15/by-date/2023-01-03.json
parking_v2/serving/current/target=y_15/by-date/2023-01-04.json
parking_v2/serving/current/target=y_15/by-date/2023-01-05.json
parking_v2/serving/current/target=y_15/by-date/2023-01-06.json
parking_v2/serving/current/target=y_15/by-date/2023-01-07.json
parking_v2/serving/current/target=y_15/by-date/2023-01-08.json
parking_v2/serving/current/target=y_15/by-date/2023-01-09.json
parking_v2/serving/current/target=y_15/by-date/2023-01-10.json
parking_v2/serving/current/target=y_15/by-date/2023-01-11.json
parking_v2/serving/current/target=y_15/by-date/2023-01-12.json
parking_v2/serving/current/target=y_15/by-date/2023-01-13.json
parking_v2/serving/current/target=y_15/by-date/2023-01-14.json
parking_v2/serving/current/target=y_15/by-date/2023-01-15.json
parking_v2/serving/current/target=y_15/by-date/2023-01-16.json
parking_v2/serving/current/target

In [9]:
build_serving_files_for_target_lowmem(
    cfg,
    cfg.input_prefix_y15,
    cfg.output_prefix_y15,
    "y_15",
    limit_files=None,
)

Found 50 prediction files for y_15
[1/50] reading parking_v2/preds/final/year=2023/target=y_15/week=2023-01-02/pred.csv.gz


KeyboardInterrupt: 

In [ ]:
build_serving_files_for_target_lowmem(
    cfg,
    cfg.input_prefix_y30,
    cfg.output_prefix_y30,
    "y_30",
    limit_files=None,
)